In [ ]:
# Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import xgboost as xgb

sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '3_utils'))

In [ ]:
# Global plot style
matplotlib.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 13,
    'axes.labelweight': 'medium',
    'xtick.labelsize': 11,
    'ytick.labelsize': 11
})

In [ ]:
# Paths
SELECTION_DIR = os.path.join('..', '4_model_training_and_evaluation', '03_xgboost', 'zone_level_tuning', 'model_selection_results')
SAVE_DIR = 'plots'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Manually selected best model per zone after reviewing top 10 candidates
zone_trials = {
    1: "trial_093.json",
    2: "trial_051.json",
    3: "trial_050.json",
    4: "trial_061.json",
    5: "trial_284.json",
    6: "trial_031.json",
    7: "trial_228.json",
    8: "trial_236.json"
}

In [ ]:
# Helper — generate concise feature names from lag params
def generate_feature_names(fire_lag, climate_lag):
    climate_features = ['Temp', 'Precip', 'Hum', 'Wind']
    lagged_fire = [f'Fire_lag{i}' for i in range(1, fire_lag + 1)]
    lagged_climate = [f'{col}_lag{i}' for col in climate_features for i in range(1, climate_lag + 1)]
    return climate_features + lagged_fire + lagged_climate

In [ ]:
# Helper — map XGBoost internal feature IDs to named features
def map_feature_names(importance_dict, feature_names):
    mapped = {}
    for f_id, importance in importance_dict.items():
        idx = int(f_id.replace('f', ''))
        if idx < len(feature_names):
            mapped[feature_names[idx]] = importance
    return mapped

In [ ]:
# Load selected model info per zone from model selection results
zone_info = {}
for zone_id, selected_filename in zone_trials.items():
    csv_path = os.path.join(SELECTION_DIR, f'zone_{zone_id}', f'zone_{zone_id}_xgboost_top_models_test_eval.csv')
    if not os.path.exists(csv_path):
        print(f"Skipping Zone {zone_id} — no model selection results found.")
        continue
    df_results = pd.read_csv(csv_path)
    row = df_results[df_results['model_filename'] == selected_filename]
    if row.empty:
        print(f"Zone {zone_id} — {selected_filename} not found in results.")
        continue
    row = row.iloc[0]
    zone_info[zone_id] = {
        'model_path': row['model_path'],
        'fire_lag': int(row['fire_lag']),
        'climate_lag': int(row['climate_lag'])
    }
    print(f"Zone {zone_id} — loaded: {selected_filename}")

In [ ]:
# Compute and save individual feature importance plot per zone
for zone_id, info in zone_info.items():
    model = xgb.Booster()
    model.load_model(info['model_path'])

    feature_names = generate_feature_names(info['fire_lag'], info['climate_lag'])
    importance_raw = model.get_score(importance_type='gain')
    importance_named = map_feature_names(importance_raw, feature_names)

    imp_df = (
        pd.DataFrame.from_dict(importance_named, orient='index', columns=['Importance'])
        .sort_values('Importance', ascending=True)
    )

    plt.figure(figsize=(8, 6))
    imp_df.tail(15).plot(kind='barh', legend=False, color='teal')
    plt.title(f"Zone {zone_id} — Top 15 Feature Importances (XGBoost)")
    plt.xlabel("Importance (gain)")
    plt.ylabel("Feature")
    plt.tight_layout()

    out_path = os.path.join(SAVE_DIR, f"zone_{zone_id}_feature_importance.png")
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"Zone {zone_id} — saved to {out_path}")

In [ ]:
# Plot all 8 zones in a 2x4 subplot grid with normalized importance
top_n = 10
cmap = cm.get_cmap("coolwarm", top_n)

fig, axes = plt.subplots(2, 4, figsize=(20, 10), constrained_layout=True)
axes = axes.flatten()

for ax, (zone_id, info) in zip(axes, zone_info.items()):
    model = xgb.Booster()
    model.load_model(info['model_path'])

    feature_names = generate_feature_names(info['fire_lag'], info['climate_lag'])
    importance_raw = model.get_score(importance_type='gain')
    importance_named = map_feature_names(importance_raw, feature_names)

    imp_df = (
        pd.DataFrame.from_dict(importance_named, orient='index', columns=['Importance'])
        .sort_values('Importance', ascending=True)
    )
    imp_df['Importance'] = imp_df['Importance'] / imp_df['Importance'].max()
    imp_df_tail = imp_df.tail(top_n)
    colors = cmap(np.linspace(0.25, 0.95, len(imp_df_tail)))

    ax.barh(imp_df_tail.index, imp_df_tail['Importance'], color=colors)
    ax.set_title(f"Zone {zone_id}", fontsize=15, weight='bold', pad=10)
    ax.tick_params(axis='x', labelsize=11)
    ax.tick_params(axis='y', labelsize=10)
    ax.set_xlim(0, 1.05)
    ax.grid(axis='x', linestyle='--', alpha=0.4)

fig.text(0.5, 0.02, "Normalized Importance (gain)", ha='center',
         fontsize=17, fontfamily='DejaVu Serif', fontweight='bold')
fig.text(0.02, 0.5, "Features", ha='center', va='center',
         rotation='vertical', fontsize=17, fontfamily='DejaVu Serif', fontweight='bold')

out_path = os.path.join(SAVE_DIR, "xgboost_zone_feature_importance_all_zones.png")
plt.savefig(out_path, dpi=600, bbox_inches='tight')
plt.close()
print(f"Saved combined plot to {out_path}")